# Background music in nine stores: pooled basket lift

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Meta-Analysis of the Difference between Means** · a worked example from the StatNex teaching guide

**The story.** A retail chain tested background music in nine stores, comparing the average basket size (\$) of shoppers on days with music and days without. Most stores saw a small, uncertain lift.

**The question.** Taking all nine stores together, does background music increase the average basket?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `03_two_sample/mean_difference_meta_analysis.csv` (9 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Store,n (music),Mean (music),SD (music),n (no music),Mean (no music),SD (no music)
1.0,Downtown,120.0,24.5,9.8,118.0,22.1,9.5
2.0,Airport,45.0,31.2,12.4,50.0,30.8,12.9
3.0,Campus,210.0,12.8,5.1,205.0,11.9,4.8
4.0,Mall East,88.0,27.4,10.2,92.0,24.0,10.6
5.0,Mall West,64.0,26.1,11.0,60.0,26.9,10.4
6.0,Station,150.0,15.3,6.2,146.0,14.1,6.0
7.0,Suburb,38.0,33.5,13.1,41.0,29.8,12.7
8.0,Riverside,55.0,21.0,8.4,52.0,20.2,8.9
9.0,Old Town,72.0,19.6,7.7,75.0,18.9,7.5

These cells save `dersimonian_laird.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile dersimonian_laird.py
"""Fixed-effect and random-effects pooling of study estimates (inverse variance,
DerSimonian-Laird).

statsmodels.stats.meta_analysis.combine_effects(effect, se**2, method_re="dl") does the
same arithmetic except one step: it does not cut tau^2 at zero. When the studies agree
more closely than chance alone would predict, Q is below its degrees of freedom and
its tau^2 comes out negative, which moves every random-effects number. A negative
variance estimate means "no detectable heterogeneity", so here it becomes 0.
"""

import numpy as np


def dersimonian_laird(effects, errors):
    """Pool estimates with standard errors `errors`; returns a dict of the results.

    Fixed effect weights each study by 1 / se^2. Random effects first adds tau^2, the
    between-study variance, to every se^2: tau^2 is the excess of Cochran's Q over its
    degrees of freedom, rescaled to the variance scale and cut at 0.
    """
    effects, errors = np.asarray(effects, float), np.asarray(errors, float)
    w = 1 / errors**2
    fixed = (w * effects).sum() / w.sum()
    q = (w * (effects - fixed) ** 2).sum()
    df = len(effects) - 1
    scale = w.sum() - (w**2).sum() / w.sum()
    tau2 = max(0.0, (q - df) / scale) if scale > 0 else 0.0
    w_random = 1 / (errors**2 + tau2)
    return {"fixed": fixed, "fixed_se": 1 / np.sqrt(w.sum()), "q": q, "df": df, "tau2": tau2,
            "random": (w_random * effects).sum() / w_random.sum(),
            "random_se": 1 / np.sqrt(w_random.sum())}

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from dersimonian_laird import dersimonian_laird

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", None)


def stars(p):
    """* for P < 0.05, ** for P < 0.01."""
    return "**" if p < 0.01 else "*" if p < 0.05 else ""

## The data

In [ ]:
# One row per study: the sample size, mean and SD of group 1, then the same three
# for group 2. A study is used only if all six are numbers, both sizes are above 0,
# n1 + n2 > 2 and the pooled SD is above 0.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["n (music)", "Mean (music)", "SD (music)", "n (no music)", "Mean (no music)", "SD (no music)"]
confidence = 95 / 100
z = stats.norm.ppf(0.5 + confidence / 2)

numbers = data[columns].apply(pd.to_numeric, errors="coerce")
n1, m1, s1, n2, m2, s2 = (numbers[c] for c in columns)
valid = (numbers.notna().all(axis=1) & (n1 > 0) & (n2 > 0) & (n1 + n2 > 2)
         & ((n1 - 1) * s1**2 + (n2 - 1) * s2**2 > 0))
cases = pd.DataFrame({"n": [valid.sum(), (~valid).sum(), len(valid)]},
                     index=["valid", "missing or invalid", "total"])
cases["share"] = cases["n"] / len(valid)
print(cases)

labels = data.loc[valid, "Store"].astype(str).tolist()
n1, m1, s1, n2, m2, s2 = (numbers.loc[valid, c].to_numpy(float) for c in columns)

## Each study: difference of means and standardised difference

In [ ]:
# Each study's difference of means has standard error pooled SD * sqrt(1/n1 + 1/n2),
# which assumes the two groups in a study are equally spread. The standardised
# difference is the difference over the pooled SD (Cohen's d, without Hedges'
# small-sample correction); its standard error grows with d itself.
pooled_sd = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
difference = m1 - m2
difference_se = pooled_sd * np.sqrt(1 / n1 + 1 / n2)
smd = difference / pooled_sd
smd_se = np.sqrt((n1 + n2) / (n1 * n2) + smd**2 / (2 * (n1 + n2)))
studies = pd.DataFrame({
    "n": n1 + n2, "difference": difference, "SE": difference_se,
    "lower": difference - z * difference_se, "upper": difference + z * difference_se,
    "pooled SD": pooled_sd, "SMD": smd, "SMD SE": smd_se,
    "SMD lower": smd - z * smd_se, "SMD upper": smd + z * smd_se}, index=labels)
print()
print(studies)

## Pooled estimates: fixed effect and random effects (DerSimonian-Laird)

In [ ]:
# The fixed-effect model assumes one true effect and weights each study by 1 / SE², so
# precise studies count most. The random-effects model (DerSimonian and Laird) lets the
# true effect vary between studies: it adds the between-study variance tau² to every
# SE² first, which evens the weights out and widens the interval. The arithmetic is in
# dersimonian_laird.py, because statsmodels' combine_effects(method_re="dl") does not
# cut a negative tau² at 0: when Q is below its degrees of freedom it returns a
# negative tau², and a random-effects estimate that gives some studies negative weight.
raw = dersimonian_laird(difference, difference_se)
standardised = dersimonian_laird(smd, smd_se)
pooled = pd.DataFrame(
    [[np.nan, raw["fixed"], raw["fixed_se"],
      np.nan, standardised["fixed"], standardised["fixed_se"]],
     [raw["tau2"], raw["random"], raw["random_se"],
      standardised["tau2"], standardised["random"], standardised["random_se"]]],
    index=["fixed effect", "random effects"],
    columns=["tau2", "difference", "SE", "SMD tau2", "SMD", "SMD SE"])
pooled.insert(3, "lower", pooled["difference"] - z * pooled["SE"])
pooled.insert(4, "upper", pooled["difference"] + z * pooled["SE"])
pooled["SMD lower"] = pooled["SMD"] - z * pooled["SMD SE"]
pooled["SMD upper"] = pooled["SMD"] + z * pooled["SMD SE"]
print()
print(pooled)

## Test of homogeneity (Cochran's Q)

In [ ]:
# Q adds up each study's squared distance from the fixed-effect estimate, in units of
# its own standard error. If every study had the same true effect, Q would average its
# degrees of freedom (studies - 1). A large P means no sign of disagreement, not proof
# of agreement: with few studies the test has little power.
homogeneity = pd.DataFrame(
    [[raw["q"], raw["df"], stats.chi2.sf(raw["q"], raw["df"])],
     [standardised["q"], standardised["df"], stats.chi2.sf(standardised["q"], standardised["df"])]],
    index=["difference", "SMD"], columns=["chi-square", "df", "P"])
print()
print(homogeneity)

## Is the pooled effect different from 0?

In [ ]:
# (estimate / SE)² against chi-square with 1 degree of freedom. It treats every person
# within a study as independent; if they were measured in clusters (days, stores), the
# true uncertainty is larger.
rows = []
for effect, result in (("difference", raw), ("SMD", standardised)):
    for model, estimate, se in (("fixed effect", result["fixed"], result["fixed_se"]),
                                ("random effects", result["random"], result["random_se"])):
        chi2 = (estimate / se) ** 2
        p = stats.chi2.sf(chi2, 1)
        rows.append([effect, model, chi2, 1, p, stars(p)])
significance = pd.DataFrame(rows, columns=["effect", "model", "chi-square", "df", "P", ""])
print()
print(significance.to_string(index=False))

## What the result says

Each store gives its own difference of means (music days minus no-music days) with a 95% confidence interval, the range of values the data agree with for the true difference. Only Mall East's interval, from \$0.36 to \$6.44, stays above zero. The other stores cannot show a lift alone.

A meta-analysis pools the stores and gives more weight to the precise ones. The pooled difference is \$1.15, with an interval from \$0.49 to \$1.81. The test of significance gives P = 0.0006, the chance of a pooled lift at least this large if music did nothing. The test of homogeneity (P = 0.71) gives no sign that the effect differs between stores, so the fixed-effect and random-effects models agree.

So music does go with larger baskets. But pooling cannot fix how the music days were chosen.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.